# Day 20 — Population-weighted load assignment

Replace the blanket `LOAD_PER_BUS_MW = 1.5` constant with a per-province target. Each distribution bus in province *P* gets `p_mw = target_mw[P] / n_dist_buses[P]`, so the per-province sum of `p_mw` matches the published peak demand for that province.

**Input:**
- [backend/data/boundaries/province_peak_targets.csv](../backend/data/boundaries/province_peak_targets.csv) — 16 rows, hand-curated rough estimates per province. Source notes per row; Phase 2 should refine with actual NGCP / DOE per-province load curves.

**Behaviour:**
- Only distribution buses get their `p_mw` rewritten — transmission roots, towers, generators, BESS, and v1_curated substations keep their existing `p_mw` (usually `NaN`).
- `q_mvar` is recomputed via the same PF=0.9 → `tan φ ≈ 0.484` rule used in Phase 1C.
- Province totals are validated against the target with a tolerance log.

**Position in pipeline:** runs *after* all distribution generation (notebooks 09 and 10) so the per-province distribution-bus count is final. Wired in as the last step of `scripts/run_phase1.py`.

In [1]:
from pathlib import Path
import math
import pandas as pd

PROC_DIR  = Path('../backend/data/processed')
BOUND_DIR = Path('../backend/data/boundaries')

PF      = 0.9
TAN_PHI = math.tan(math.acos(PF))   # ≈ 0.484

In [2]:
buses    = pd.read_csv(PROC_DIR / 'buses.csv')
targets  = pd.read_csv(BOUND_DIR / 'province_peak_targets.csv')
print(f'buses: {len(buses)} rows')
print(f'targets: {len(targets)} provinces, sum = {targets["peak_mw"].sum()} MW')
print()
print(targets[['province', 'peak_mw']].to_string(index=False))

buses: 2959 rows
targets: 16 provinces, sum = 2282 MW

         province  peak_mw
             Cebu      720
Negros Occidental      320
           Iloilo      250
            Leyte      200
  Negros Oriental      160
            Bohol      120
            Samar       90
            Capiz       80
            Aklan       75
   Northern Samar       65
          Antique       55
    Eastern Samar       45
   Southern Leyte       40
         Guimaras       25
          Biliran       22
         Siquijor       15


## §1 — Pre-run snapshot per province

In [3]:
dist_mask = buses['bus_type'] == 'distribution'
pre = buses.groupby('province').agg(
    total_load_before_mw=('p_mw', 'sum'),
    n_dist_buses=('bus_type', lambda s: (s == 'distribution').sum()),
).reset_index()
print(pre.to_string(index=False))
print(f'\nVisayas total before: {pre["total_load_before_mw"].sum():.0f} MW')

         province  total_load_before_mw  n_dist_buses
            Aklan                  66.0            44
          Antique                  36.0            24
          Biliran                  27.0            18
            Bohol                 352.5           235
            Capiz                  36.0            24
             Cebu                1381.5           921
    Eastern Samar                  72.0            48
         Guimaras                  30.0            20
           Iloilo                 225.0           150
            Leyte                 664.5           443
Negros Occidental                 435.0           290
  Negros Oriental                 366.0           244
   Northern Samar                  99.0            66
            Samar                 223.5           149
         Siquijor                  33.0            22
   Southern Leyte                  73.5            49

Visayas total before: 4120 MW


## §2 — Scale each distribution bus to the per-province target

If a province has zero distribution buses (none in the current pipeline output), it gets skipped with a warning.

In [4]:
target_map = dict(zip(targets['province'], targets['peak_mw']))

# Compute per-bus load: target_mw / n_dist_buses
dist_counts = buses[dist_mask].groupby('province').size().rename('n_dist')

per_bus_load = {}
for prov in buses['province'].dropna().unique():
    n = int(dist_counts.get(prov, 0))
    tgt = target_map.get(prov)
    if n == 0:
        per_bus_load[prov] = 0.0
        if tgt is not None and tgt > 0:
            print(f'  warning: {prov} has target {tgt} MW but 0 distribution buses')
    elif tgt is None:
        per_bus_load[prov] = 0.0
        print(f'  warning: {prov} has {n} distribution buses but no target — setting p_mw=0')
    else:
        per_bus_load[prov] = tgt / n

# Apply: only to distribution buses
buses_new = buses.copy()
new_p_mw = buses_new.apply(
    lambda r: per_bus_load[r['province']] if r['bus_type'] == 'distribution' else r['p_mw'],
    axis=1,
)
buses_new['p_mw']   = new_p_mw
buses_new['q_mvar'] = new_p_mw.where(buses_new['bus_type'] == 'distribution',
                                       buses_new['q_mvar']) * TAN_PHI
# For non-distribution rows where p_mw is unchanged, preserve original q_mvar
buses_new.loc[buses_new['bus_type'] != 'distribution', 'q_mvar'] = buses.loc[
    buses['bus_type'] != 'distribution', 'q_mvar']

print('\nPer-bus load (MW) by province:')
for prov in sorted(per_bus_load):
    n = int(dist_counts.get(prov, 0))
    if n:
        print(f'  {prov:<22s} {per_bus_load[prov]:6.3f} MW/bus × {n:4d} buses = {per_bus_load[prov]*n:6.1f} MW')


Per-bus load (MW) by province:
  Aklan                   1.705 MW/bus ×   44 buses =   75.0 MW
  Antique                 2.292 MW/bus ×   24 buses =   55.0 MW
  Biliran                 1.222 MW/bus ×   18 buses =   22.0 MW
  Bohol                   0.511 MW/bus ×  235 buses =  120.0 MW
  Capiz                   3.333 MW/bus ×   24 buses =   80.0 MW
  Cebu                    0.782 MW/bus ×  921 buses =  720.0 MW
  Eastern Samar           0.938 MW/bus ×   48 buses =   45.0 MW
  Guimaras                1.250 MW/bus ×   20 buses =   25.0 MW
  Iloilo                  1.667 MW/bus ×  150 buses =  250.0 MW
  Leyte                   0.451 MW/bus ×  443 buses =  200.0 MW
  Negros Occidental       1.103 MW/bus ×  290 buses =  320.0 MW
  Negros Oriental         0.656 MW/bus ×  244 buses =  160.0 MW
  Northern Samar          0.985 MW/bus ×   66 buses =   65.0 MW
  Samar                   0.604 MW/bus ×  149 buses =   90.0 MW
  Siquijor                0.682 MW/bus ×   22 buses =   15.0 MW
  Southe

## §3 — Validate against the targets

In [5]:
post = buses_new.groupby('province').agg(
    total_load_after_mw=('p_mw', 'sum'),
    n_dist_buses=('bus_type', lambda s: (s == 'distribution').sum()),
).reset_index()

cmp = pre.merge(post, on='province', suffixes=('', '_post'))
cmp = cmp.merge(targets[['province', 'peak_mw']], on='province', how='left')
cmp['target_mw']    = cmp['peak_mw']
cmp['pct_of_target']= 100 * cmp['total_load_after_mw'] / cmp['target_mw']
cmp = cmp[['province', 'n_dist_buses', 'total_load_before_mw',
            'total_load_after_mw', 'target_mw', 'pct_of_target']].sort_values(
                'total_load_after_mw', ascending=False)
print(cmp.round(1).to_string(index=False))

total_after = post['total_load_after_mw'].sum()
target_total = targets['peak_mw'].sum()
print(f'\nVisayas total after:  {total_after:.0f} MW')
print(f'Target total:         {target_total} MW')
print(f'Real Visayas peak:    ~2 200 MW')

         province  n_dist_buses  total_load_before_mw  total_load_after_mw  target_mw  pct_of_target
             Cebu           921                1381.5                720.0        720          100.0
Negros Occidental           290                 435.0                320.0        320          100.0
           Iloilo           150                 225.0                250.0        250          100.0
            Leyte           443                 664.5                200.0        200          100.0
  Negros Oriental           244                 366.0                160.0        160          100.0
            Bohol           235                 352.5                120.0        120          100.0
            Samar           149                 223.5                 90.0         90          100.0
            Capiz            24                  36.0                 80.0         80          100.0
            Aklan            44                  66.0                 75.0         75      

## §4 — Save

In [6]:
# Schema sanity: all original columns preserved, only p_mw/q_mvar changed for distribution rows
assert list(buses_new.columns) == list(buses.columns)
assert buses_new['bus_id'].is_unique

buses_new.to_csv(PROC_DIR / 'buses.csv', index=False)
cmp.to_csv(PROC_DIR / 'load_assignment_summary.csv', index=False)
print(f'Wrote {PROC_DIR / "buses.csv"} ({len(buses_new)} rows)')
print(f'Wrote {PROC_DIR / "load_assignment_summary.csv"} ({len(cmp)} rows)')

Wrote ../backend/data/processed/buses.csv (2959 rows)
Wrote ../backend/data/processed/load_assignment_summary.csv (16 rows)
